# Solution C3 · Too many features: when the training R² lies

**Type**: Break the model

## Objective

Add pure noise features one at a time to a model with 60 observations, until the features reach 59:
measure training R², adjusted R² and cross-validated R², and watch Ridge hold up where OLS collapses.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
n = 60
x_true = rng.normal(size=n)
noise = rng.normal(size=(n, 58))
y = 2 * x_true + rng.normal(size=n)
cv = KFold(5, shuffle=True, random_state=0)
ks = np.arange(0, 59)

## Commented solution

In [ ]:
r2_train = np.empty(len(ks))
r2_adj = np.empty(len(ks))
r2_cv = np.empty(len(ks))
for i, k in enumerate(ks):
    X_k = np.column_stack([x_true, noise[:, :k]])
    model = LinearRegression().fit(X_k, y)
    r2_train[i] = model.score(X_k, y)
    p = k + 1
    r2_adj[i] = 1 - (1 - r2_train[i]) * (n - 1) / (n - p - 1) if n - p - 1 > 0 else np.nan
    r2_cv[i] = cross_val_score(LinearRegression(), X_k, y, cv=cv, scoring="r2").mean()

In [ ]:
best_k_cv = int(np.argmax(r2_cv))
negative_k = int(np.argmax(r2_cv < 0))
print("best k in cross-validation:", best_k_cv, "| first k with a negative cv R2:", negative_k)

In [ ]:
r2_cv_ridge = np.empty(len(ks))
for i, k in enumerate(ks):
    X_k = np.column_stack([x_true, noise[:, :k]])
    ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 20)))
    r2_cv_ridge[i] = cross_val_score(ridge, X_k, y, cv=cv, scoring="r2").mean()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(ks, r2_train, label="training R2 (OLS)")
ax.plot(ks, r2_adj, label="adjusted R2 (OLS)")
ax.plot(ks, r2_cv, label="cross-validated R2 (OLS)")
ax.plot(ks, r2_cv_ridge, label="cross-validated R2 (Ridge)")
ax.axhline(0, color="k", lw=0.8)
ax.set(xlabel="number of pure noise features added", ylabel="R2", ylim=(-3, 1.05)); ax.legend();

In [ ]:
print(f"with k = 58: train R2 {r2_train[-1]:.3f} | cv R2 OLS {r2_cv[-1]:.2f} | cv R2 Ridge {r2_cv_ridge[-1]:.2f}")

## Check

In [ ]:
with verify():
    check(np.all(np.diff(r2_train) >= -1e-9), "the training R2 never decreases when you add features",
          "r2_train should be non-decreasing in k")
    check(r2_train[-1] > 0.999, "with 59 features and 60 observations the training R2 is 1",
          "with k = 58 the model interpolates the data: r2_train should be about 1")
    check(np.isnan(r2_adj[-1]) and not np.isnan(r2_adj[:-1]).any(),
          "the adjusted R2 is undefined only for k = 58", "use np.nan when n - p - 1 is zero, and only then")
    check(np.all(r2_adj[:-1] <= r2_train[:-1] + 1e-12), "the adjusted R2 never exceeds the training one",
          "r2_adj must be less than or equal to r2_train")
    check(best_k_cv <= 10, "cross-validation prefers a model with few features",
          "best_k_cv should be low (the additional features are just noise)")
    check(r2_cv.min() < -0.5, "cross-validation becomes very negative with too many features",
          "with many noise features r2_cv should drop below -0.5")
    check(negative_k == int(np.argmax(r2_cv < 0)) and negative_k < 40, "negative_k is the first k with a negative cv R2",
          "negative_k must be the first index for which r2_cv is less than zero")
    check(r2_cv_ridge[58] > r2_cv[58] + 0.5, "Ridge holds up much better than OLS with 59 features",
          "with k = 58 Ridge should have a much better cv R2 than OLS")
done()

## What you learned

- `r2_train` always goes up, to 1: with 59 parameters and 60 observations the model interpolates the data. It is not a measure
  of quality, it is a measure of **how free the model is**.
- The adjusted R² penalizes the number of features, but it remains a correction computed on the training set and becomes
  unstable near p = n. Cross-validation measures the right thing: performance on unseen data.
- The collapse of OLS comes early: already with 28 noise features, fewer than half the observations, the cross-validated
  R² is negative, that is the model is worse than the mean. Ridge, by constraining the coefficients, stays much more stable: it is not magic, it is the same idea
  as the bias-variance tradeoff of theory lesson 1.1.

## Common pitfall

Comparing models with a different number of features using the training R², or reporting the training R² as the
"quality of the model" in a report. With p close to n it is a number that contains no information about
generalization.

## Interview question

> *Click on a question to show its answer.*

<details><summary>A colleague reports R² = 0.96 with 80 observations and 70 features. How do you react?</summary>

I ask for the cross-validation before any conclusion: with p close to n the training R² is close to 1 by
construction, even on pure noise. I would compute the adjusted R² (or better the cross-validated one) and consider
a reduction of the features or a regularization, and I would repeat everything on a held-out test set.

</details>